# 03 Training Pipeline

This notebook runs the LoRA experiments for TinyLlama and saves structured outputs under `./results/`.

It is designed for Google Colab GPU and supports:
- a shared train/validation split
- response-only loss masking
- LoRA rank sweeps for `r = 2, 4, 8, 16`
- an optional full fine-tuning baseline attempt
- automatic logging of trainable parameters, evaluation loss, perplexity, wall-clock time, and peak GPU memory
- lightweight result saving without full checkpoints for every run


In [ ]:
from pathlib import Path

repo_dir = Path('/content/DSA5204')
if not repo_dir.exists():
    !git clone https://github.com/NomadZhang/DSA5204.git /content/DSA5204

%cd /content/DSA5204
!pip install -q "transformers>=4.57.0" "datasets>=2.19.0" accelerate pandas matplotlib sentencepiece seaborn


In [ ]:
import inspect
import time
from pathlib import Path

import pandas as pd
import torch
import transformers
from transformers import Trainer, TrainingArguments, default_data_collator

from src.experiment_utils import (
    DEFAULT_SAMPLE_PROMPTS,
    alpha_for_rank,
    build_model,
    generate_samples,
    load_and_prepare_datasets,
    load_tokenizer,
    parameter_statistics,
    peak_gpu_memory_mb,
    reset_peak_gpu_memory,
    safe_perplexity,
    set_all_seeds,
    write_json,
)

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATA_PATH = "./data/train.jsonl"
RESULTS_DIR = Path("./results")
SEED = 42
VALIDATION_SIZE = 0.1
MAX_LENGTH = 256
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
MAX_STEPS = 50
EVAL_STEPS = 25
LOGGING_STEPS = 10
RUN_FULL_FINE_TUNING_BASELINE = True
RANKS = [2, 4, 8, 16]
SAMPLE_PROMPTS = DEFAULT_SAMPLE_PROMPTS

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = None

set_all_seeds(SEED)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Transformers version: {transformers.__version__}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory total: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.2f} GB")


In [ ]:
tokenizer = load_tokenizer(MODEL_ID)
raw_splits, tokenized_splits = load_and_prepare_datasets(
    DATA_PATH,
    tokenizer,
    max_length=MAX_LENGTH,
    validation_size=VALIDATION_SIZE,
    seed=SEED,
)

print(raw_splits)
print(tokenized_splits)
display(raw_splits["train"].select(range(3)).to_pandas())


In [ ]:
def build_run_configs():
    configs = []
    if RUN_FULL_FINE_TUNING_BASELINE:
        configs.append(
            {
                "run_name": "full_ft_baseline",
                "method": "full_ft",
                "use_lora": False,
                "r": None,
                "alpha": None,
            }
        )

    for rank in RANKS:
        configs.append(
            {
                "run_name": f"lora_r{rank}",
                "method": "lora",
                "use_lora": True,
                "r": rank,
                "alpha": alpha_for_rank(rank),
            }
        )
    return configs


def make_training_args(run_name):
    training_kwargs = {
        "output_dir": str(RESULTS_DIR / run_name / "trainer_output"),
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "max_steps": MAX_STEPS,
        "eval_steps": EVAL_STEPS,
        "save_strategy": "no",
        "logging_steps": LOGGING_STEPS,
        "report_to": "none",
        "remove_unused_columns": False,
        "fp16": False,
        "dataloader_pin_memory": torch.cuda.is_available(),
        "seed": SEED,
    }

    signature = inspect.signature(TrainingArguments.__init__).parameters
    if "overwrite_output_dir" in signature:
        training_kwargs["overwrite_output_dir"] = True
    if "evaluation_strategy" in signature:
        training_kwargs["evaluation_strategy"] = "steps"
    elif "eval_strategy" in signature:
        training_kwargs["eval_strategy"] = "steps"
    if "bf16" in signature:
        training_kwargs["bf16"] = False

    return TrainingArguments(**training_kwargs)


def run_experiment(config):
    run_name = config["run_name"]
    run_dir = RESULTS_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n===== Running {run_name} =====")
    reset_peak_gpu_memory()
    set_all_seeds(SEED)

    model = None
    trainer = None
    status = "completed"
    error_message = None
    eval_metrics = {}
    train_output = None
    train_runtime_sec = 0.0
    global_step = 0
    stats = {
        "total_params": None,
        "trainable_params": None,
        "frozen_params": None,
        "trainable_ratio": None,
    }
    gpu_memory_after_load_mb = None

    try:
        model = build_model(
            MODEL_ID,
            use_lora=config["use_lora"],
            r=config["r"],
            alpha=config["alpha"],
            torch_dtype=DTYPE,
        )
        model.to(DEVICE)
        model.config.use_cache = False
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()

        stats = parameter_statistics(model)
        gpu_memory_after_load_mb = torch.cuda.memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

        trainer = Trainer(
            model=model,
            args=make_training_args(run_name),
            train_dataset=tokenized_splits["train"],
            eval_dataset=tokenized_splits["test"],
            data_collator=default_data_collator,
        )

        start_time = time.time()
        train_output = trainer.train()
        train_runtime_sec = time.time() - start_time
        eval_metrics = trainer.evaluate()
        global_step = trainer.state.global_step

        model.config.use_cache = True
        samples = generate_samples(model, tokenizer, SAMPLE_PROMPTS, device=DEVICE)
        write_json(run_dir / "sample_generations.json", {"samples": samples})

    except RuntimeError as exc:
        train_runtime_sec = time.time() - start_time if 'start_time' in locals() else 0.0
        status = "oom" if "out of memory" in str(exc).lower() else "failed"
        error_message = str(exc)
        if model is not None:
            stats = parameter_statistics(model)
        gpu_memory_after_load_mb = torch.cuda.memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"Run {run_name} ended with status={status}: {error_message}")

    peak_memory = peak_gpu_memory_mb()
    eval_loss = eval_metrics.get("eval_loss")
    perplexity = safe_perplexity(eval_loss) if eval_loss is not None else None
    train_loss = train_output.training_loss if train_output is not None else None
    step_time_sec = train_runtime_sec / global_step if global_step else None

    summary = {
        "run_name": run_name,
        "status": status,
        "method": config["method"],
        "r": config["r"],
        "alpha": config["alpha"],
        "seed": SEED,
        "max_length": MAX_LENGTH,
        "max_steps": MAX_STEPS,
        "learning_rate": LEARNING_RATE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
        **stats,
        "gpu_memory_after_load_mb": gpu_memory_after_load_mb,
        "peak_gpu_memory_mb": peak_memory,
        "train_runtime_sec": train_runtime_sec,
        "step_time_sec": step_time_sec,
        "global_step": global_step,
        "train_loss": train_loss,
        "eval_loss": eval_loss,
        "perplexity": perplexity,
        "error_message": error_message,
    }

    write_json(run_dir / "metrics_summary.json", summary)

    if trainer is not None:
        del trainer
    if model is not None:
        del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


run_summaries = [run_experiment(config) for config in build_run_configs()]
summary_df = pd.DataFrame(run_summaries)
summary_df.to_csv(RESULTS_DIR / "experiment_summary.csv", index=False)
display(summary_df)


In [ ]:
summary_df[[
    "run_name",
    "status",
    "method",
    "r",
    "trainable_params",
    "trainable_ratio",
    "peak_gpu_memory_mb",
    "eval_loss",
    "perplexity",
    "step_time_sec",
]].sort_values(["method", "r"], na_position="first")
